In [1]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", None)

In [2]:
with open(r"/home/user/Downloads/bahrainfinder_2026_07_04.json", "r", encoding="utf-8") as f:
    for i in range(5):
        print(f.readline())

{"reference_number": "1252", "url": "https://bahrainfinder.bh/en/property/villa-for-sale-in-jid-al-haj-11585.html", "broker_display_name": "LAKUM Real Estates", "broker": "LAKUM REAL ESTATES", "category": "sale", "category_url": "https://bahrainfinder.bh/en/for-sale/", "title": "Residential Villa For Sale", "description": "Residential Villa for Sale in Jid Al HajArea: 254.5 sqmBuilt-up size: 230 sqmBathrooms: 2Halls: 1Floors: 2Furnishing: FurnishedCondition: NewParking spaces: 2Price: 129,000 BHDRoads: 1Features & Amenities: Garden, Closed Kitchen, Swimming PoolFor more information, do not hesitate to contact me now | Manaf Tareef – LAKUM Real Estates", "location": "Jid Al Haj", "price": "129,000", "currency": "BHD", "price_per": "", "bedrooms": "", "bathrooms": "2", "furnished": "Furnished", "rera_permit_number": "", "dtcm_licence": "", "scraped_ts": "2026-07-27", "amenities": "Garden, Closed Kitchen, Swimming Pool", "details": "254.5 m²230 m²", "agent_name": "Manaf Tareef", "number_o

In [4]:
import pandas as pd
import json

# ============================================================
# 1. FILE CONFIGURATION
# ============================================================

file_path = "/home/user/Downloads/bayut_uae_2026_10_15.json"
chunk_size = 10_000

# ============================================================
# 2. EXPECTED FIELDS
# ============================================================

expected_fields = [
    "reference_number", "id", "url", "broker_display_name", "broker",
    "category", "category_url", "title", "description", "location",
    "price", "currency", "price_per", "bedrooms", "bathrooms",
    "furnished", "rera_permit_number", "dtcm_licence", "scraped_ts",
    "amenities", "details", "agent_name", "number_of_photos",
    "user_id", "phone_number", "date", "iteration_number",
    "rera_registration_number", "ded_license_number", "rera_length",
    "latitude", "longitude", "listing_id", "package_type", "locality",
    "object_id", "completion_status", "ad_type", "verified",
    "sub_category_1", "sub_category_2", "property_type", "depth",
    "published_at", "listing_availability", "brn_registration_number",
    "orn_number", "bln_number", "adm_number", "3d_live", "3d_image",
    "2d_image", "ded_number_2", "ded_number_3", "ded_number_4",
    "trucheck_on", "email", "trubroker", "rating", "no_of_reviews",
    "property_of_the_week", "quality_lister", "trakheesi_url",
    "floorplan_type_identifier", "floorplans_total", "floorplan_id",
    "floorplan_raw"
]

expected_set = set(expected_fields)

# ============================================================
# 3. INITIALIZE TRACKING VARIABLES
# ============================================================

actual_fields = set()
total_records = 0

# Count records where each expected field is absent
missing_count = {field: 0 for field in expected_fields}

# Store sample record numbers for missing fields
missing_samples = {field: [] for field in expected_fields}
max_samples = 5

# ============================================================
# 4. PROCESS JSON FILE IN CHUNKS
# ============================================================

for chunk_no, chunk in enumerate(
    pd.read_json(file_path, lines=True, chunksize=chunk_size),
    start=1
):
    print(f"Processing chunk {chunk_no}: {chunk.shape}")

    total_records += len(chunk)

    # Track every field found in the file
    actual_fields.update(chunk.columns)

    # Check which expected fields are absent from each record
    for field in expected_fields:

        if field not in chunk.columns:
            # The field is absent from every record in this chunk
            missing_count[field] += len(chunk)

            for idx in range(min(max_samples, len(chunk))):
                if len(missing_samples[field]) < max_samples:
                    missing_samples[field].append(
                        total_records - len(chunk) + idx + 1
                    )

        else:
            # A column exists, but individual JSON records may omit it.
            # pandas represents those absent values as NaN.
            absent_mask = chunk[field].isna()
            absent_rows = absent_mask[absent_mask].index

            missing_count[field] += len(absent_rows)

            for idx in absent_rows:
                if len(missing_samples[field]) < max_samples:
                    missing_samples[field].append(
                        total_records - len(chunk)
                        + chunk.index.get_loc(idx) + 1
                    )

    print(f"Total records processed: {total_records:,}")

# ============================================================
# 5. COMPARE EXPECTED AND ACTUAL FIELDS
# ============================================================

missing_fields = sorted(expected_set - actual_fields)
extra_fields = sorted(actual_fields - expected_set)

# ============================================================
# 6. PRINT FIELD VALIDATION REPORT
# ============================================================

print("\n" + "=" * 60)
print("             JSON FIELD VALIDATION REPORT")
print("=" * 60)

print(f"File: {file_path}")
print(f"Total records: {total_records:,}")
print(f"Expected fields: {len(expected_fields)}")
print(f"Actual distinct fields: {len(actual_fields)}")

print("\n--- MISSING FIELDS (Absent from the entire file) ---")
print(f"Count: {len(missing_fields)}")

if missing_fields:
    for field in missing_fields:
        print(f"❌ {field}")
else:
    print("No missing fields.")

print("\n--- EXTRA FIELDS ---")
print(f"Count: {len(extra_fields)}")

if extra_fields:
    for field in extra_fields:
        print(f"⚠️ {field}")
else:
    print("No extra fields.")

print("\n--- RECORD-LEVEL FIELD ABSENCE ---")
print("Field | Records Missing Field | Sample Record Numbers")

for field in expected_fields:
    if missing_count[field] > 0:
        print(
            f"{field} | {missing_count[field]:,} | "
            f"{missing_samples[field]}"
        )

print("\n--- FINAL STATUS ---")

if not missing_fields and not extra_fields:
    print("PASS: Expected and actual field names match.")
else:
    print("FAIL: Missing or extra field names detected.")

if any(count > 0 for count in missing_count.values()):
    print("WARNING: Some records have absent or null field values.")
else:
    print("All expected fields have non-null values in every record.")

Processing chunk 1: (10000, 67)
Total records processed: 10,000
Processing chunk 2: (10000, 67)
Total records processed: 20,000
Processing chunk 3: (10000, 67)
Total records processed: 30,000
Processing chunk 4: (10000, 67)
Total records processed: 40,000
Processing chunk 5: (10000, 67)
Total records processed: 50,000
Processing chunk 6: (10000, 67)
Total records processed: 60,000
Processing chunk 7: (10000, 67)
Total records processed: 70,000
Processing chunk 8: (10000, 67)
Total records processed: 80,000
Processing chunk 9: (10000, 67)
Total records processed: 90,000
Processing chunk 10: (10000, 67)
Total records processed: 100,000
Processing chunk 11: (10000, 67)
Total records processed: 110,000
Processing chunk 12: (10000, 67)
Total records processed: 120,000
Processing chunk 13: (10000, 67)
Total records processed: 130,000
Processing chunk 14: (10000, 67)
Total records processed: 140,000
Processing chunk 15: (10000, 67)
Total records processed: 150,000
Processing chunk 16: (10000,

In [6]:
import pandas as pd
import os

# ============================================================
# 1. FILE CONFIGURATION
# ============================================================

file_path = "/home/user/Downloads/bayut_uae_2026_10_15.json"
chunk_size = 10_000
output_dir = "/home/user/Downloads/bayut_qa_report"

os.makedirs(output_dir, exist_ok=True)

# Fields for value counts
value_count_fields = [
    "scraped_ts",
    "date",
    "iteration_number",
    "price_per",
    "furnished",
    "category",
    "sub_category_1",
    "completion_status",
    "package_type",
    "currency"
]

# ============================================================
# 2. INITIALIZE TRACKING
# ============================================================

total_records = 0
field_counts = {field: {} for field in value_count_fields}

# Exact duplicate detection across all columns:
# Store each unique full-record hash and its occurrence count.
record_hash_counts = {}

# ============================================================
# 3. PROCESS FILE IN CHUNKS
# ============================================================

for chunk_no, chunk in enumerate(
    pd.read_json(file_path, lines=True, chunksize=chunk_size),
    start=1
):
    print(f"Processing chunk {chunk_no}: {chunk.shape}")

    total_records += len(chunk)

    # --------------------------------------------------------
    # A. VALUE COUNTS
    # --------------------------------------------------------

    for field in value_count_fields:
        if field not in chunk.columns:
            continue

        counts = chunk[field].value_counts(dropna=False)

        for value, count in counts.items():
            # Convert values into dictionary-safe keys
            key = repr(value)
            field_counts[field][key] = (
                field_counts[field].get(key, 0) + int(count)
            )

    # --------------------------------------------------------
    # B. FULL-RECORD DUPLICATE CHECK
    # --------------------------------------------------------

    # Hash complete rows, including all columns in this chunk.
    # JSON serialization handles list/dict values consistently.
    normalized = chunk.copy()

    for col in normalized.columns:
        normalized[col] = normalized[col].map(
            lambda x: str(x) if isinstance(x, (dict, list)) else x
        )

    row_hashes = pd.util.hash_pandas_object(
        normalized, index=False
    )

    for h, count in row_hashes.value_counts().items():
        key = int(h)
        record_hash_counts[key] = (
            record_hash_counts.get(key, 0) + int(count)
        )

# ============================================================
# 4. DUPLICATE SUMMARY
# ============================================================

duplicate_groups = sum(
    1 for count in record_hash_counts.values() if count > 1
)

duplicate_records = sum(
    count - 1
    for count in record_hash_counts.values()
    if count > 1
)

print("\n" + "=" * 60)
print("ENTIRE DATASET UNIQUENESS")
print("=" * 60)
print(f"Total records: {total_records:,}")
print(f"Unique record-hash groups: {len(record_hash_counts):,}")
print(f"Duplicate groups: {duplicate_groups:,}")
print(f"Duplicate records beyond first occurrence: {duplicate_records:,}")

if duplicate_records == 0:
    print("Result: No duplicate records detected.")
else:
    print("Result: Duplicate records detected.")

# ============================================================
# 5. SAVE VALUE COUNTS
# ============================================================

for field, counts in field_counts.items():
    report = pd.DataFrame(
        [
            {"value": value, "count": count}
            for value, count in counts.items()
        ]
    )

    if not report.empty:
        report = report.sort_values("count", ascending=False)

    output_path = os.path.join(
        output_dir, f"{field}_value_counts.csv"
    )
    report.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")

print("\nAll checks completed.")

Processing chunk 1: (10000, 67)
Processing chunk 2: (10000, 67)
Processing chunk 3: (10000, 67)
Processing chunk 4: (10000, 67)
Processing chunk 5: (10000, 67)
Processing chunk 6: (10000, 67)
Processing chunk 7: (10000, 67)
Processing chunk 8: (10000, 67)
Processing chunk 9: (10000, 67)
Processing chunk 10: (10000, 67)
Processing chunk 11: (10000, 67)
Processing chunk 12: (10000, 67)
Processing chunk 13: (10000, 67)
Processing chunk 14: (10000, 67)
Processing chunk 15: (10000, 67)
Processing chunk 16: (10000, 67)
Processing chunk 17: (10000, 67)
Processing chunk 18: (10000, 67)
Processing chunk 19: (10000, 67)
Processing chunk 20: (10000, 67)
Processing chunk 21: (10000, 67)
Processing chunk 22: (10000, 67)
Processing chunk 23: (10000, 67)
Processing chunk 24: (10000, 67)
Processing chunk 25: (10000, 67)
Processing chunk 26: (10000, 67)
Processing chunk 27: (10000, 67)
Processing chunk 28: (10000, 67)
Processing chunk 29: (10000, 67)
Processing chunk 30: (10000, 67)
Processing chunk 31

In [7]:
import pandas as pd

file_path = "/home/user/Downloads/bayut_uae_2026_10_15.json"
chunk_size = 10_000

total_records = 0
empty_counts = {}
all_fields = set()

for chunk_no, chunk in enumerate(
    pd.read_json(file_path, lines=True, chunksize=chunk_size),
    start=1
):
    total_records += len(chunk)
    all_fields.update(chunk.columns)

    for col in chunk.columns:
        # Count nulls, empty strings, and whitespace-only strings
        empty_mask = (
            chunk[col].isna()
            | chunk[col].map(
                lambda x: isinstance(x, str) and x.strip() == ""
            )
        )

        empty_counts[col] = (
            empty_counts.get(col, 0) + int(empty_mask.sum())
        )

    print(f"Processed chunk {chunk_no}")

print("\n========== COMPLETELY EMPTY FIELDS ==========")
print(f"Total records: {total_records:,}")

completely_empty = [
    field for field in sorted(all_fields)
    if empty_counts.get(field, 0) == total_records
]

if completely_empty:
    for field in completely_empty:
        print(f"❌ {field}")
    print(f"\nTotal completely empty fields: {len(completely_empty)}")
else:
    print("No completely empty fields found.")

Processed chunk 1
Processed chunk 2
Processed chunk 3
Processed chunk 4
Processed chunk 5
Processed chunk 6
Processed chunk 7
Processed chunk 8
Processed chunk 9
Processed chunk 10
Processed chunk 11
Processed chunk 12
Processed chunk 13
Processed chunk 14
Processed chunk 15
Processed chunk 16
Processed chunk 17
Processed chunk 18
Processed chunk 19
Processed chunk 20
Processed chunk 21
Processed chunk 22
Processed chunk 23
Processed chunk 24
Processed chunk 25
Processed chunk 26
Processed chunk 27
Processed chunk 28
Processed chunk 29
Processed chunk 30
Processed chunk 31
Processed chunk 32
Processed chunk 33
Processed chunk 34
Processed chunk 35
Processed chunk 36
Processed chunk 37
Processed chunk 38
Processed chunk 39
Processed chunk 40
Processed chunk 41
Processed chunk 42
Processed chunk 43
Processed chunk 44
Processed chunk 45
Processed chunk 46
Processed chunk 47
Processed chunk 48
Processed chunk 49
Processed chunk 50
Processed chunk 51
Processed chunk 52
Processed chunk 53
Pr

In [8]:
import pandas as pd

# File configuration
file_path = "/home/user/Downloads/bayut_uae_2026_10_15.json"
chunk_size = 10_000

total_records = 0
same_values = 0
different_values = 0
missing_values = 0
broker_not_capital = 0

different_samples = []
capital_samples = []
max_samples = 10

for chunk_no, chunk in enumerate(
    pd.read_json(file_path, lines=True, chunksize=chunk_size),
    start=1
):
    print(f"Processing chunk {chunk_no}: {chunk.shape}")
    total_records += len(chunk)

    # Check required columns
    required = ["broker", "broker_display_name"]
    if not all(col in chunk.columns for col in required):
        raise ValueError(f"Missing required columns: {required}")

    broker = chunk["broker"]
    display = chunk["broker_display_name"]

    # Check missing values
    missing_mask = broker.isna() | display.isna()
    missing_values += int(missing_mask.sum())

    # Compare values, ignoring leading/trailing spaces
    broker_clean = broker.astype("string").str.strip()
    display_clean = display.astype("string").str.strip()

    same_mask = (
        broker_clean.eq(display_clean).fillna(False)
        & ~missing_mask
    )
    different_mask = ~same_mask & ~missing_mask

    same_values += int(same_mask.sum())
    different_values += int(different_mask.sum())

    # Capture examples where values differ
    if len(different_samples) < max_samples:
        sample = chunk.loc[
            different_mask, ["broker", "broker_display_name"]
        ].head(max_samples - len(different_samples))

        different_samples.extend(sample.to_dict("records"))

    # Check whether broker is uppercase
    # Ignore nulls and non-string values
    string_mask = broker.map(lambda x: isinstance(x, str))
    uppercase_mask = broker.map(
        lambda x: isinstance(x, str) and x == x.upper()
    )

    not_capital_mask = string_mask & ~uppercase_mask
    broker_not_capital += int(not_capital_mask.sum())

    # Capture examples where broker is not uppercase
    if len(capital_samples) < max_samples:
        sample = chunk.loc[
            not_capital_mask, ["broker", "broker_display_name"]
        ].head(max_samples - len(capital_samples))

        capital_samples.extend(sample.to_dict("records"))

# Final report
print("\n" + "=" * 60)
print("BROKER FIELD QA REPORT")
print("=" * 60)

print(f"Total records: {total_records:,}")

print("\n1. broker vs broker_display_name")
print(f"Same values: {same_values:,}")
print(f"Different values: {different_values:,}")
print(f"Records with missing values: {missing_values:,}")

if different_samples:
    print("\nSample mismatched values:")
    print(pd.DataFrame(different_samples).to_string(index=False))

print("\n2. broker uppercase validation")
print(f"Non-uppercase string values: {broker_not_capital:,}")

if capital_samples:
    print("\nSample non-uppercase values:")
    print(pd.DataFrame(capital_samples).to_string(index=False))

print("\nFINAL STATUS")
if different_values == 0 and missing_values == 0 and broker_not_capital == 0:
    print("PASS: Both fields match and broker is uppercase.")
else:
    print("FAIL: Differences, missing values, or non-uppercase values found.")

Processing chunk 1: (10000, 67)
Processing chunk 2: (10000, 67)
Processing chunk 3: (10000, 67)
Processing chunk 4: (10000, 67)
Processing chunk 5: (10000, 67)
Processing chunk 6: (10000, 67)
Processing chunk 7: (10000, 67)
Processing chunk 8: (10000, 67)
Processing chunk 9: (10000, 67)
Processing chunk 10: (10000, 67)
Processing chunk 11: (10000, 67)
Processing chunk 12: (10000, 67)
Processing chunk 13: (10000, 67)
Processing chunk 14: (10000, 67)
Processing chunk 15: (10000, 67)
Processing chunk 16: (10000, 67)
Processing chunk 17: (10000, 67)
Processing chunk 18: (10000, 67)
Processing chunk 19: (10000, 67)
Processing chunk 20: (10000, 67)
Processing chunk 21: (10000, 67)
Processing chunk 22: (10000, 67)
Processing chunk 23: (10000, 67)
Processing chunk 24: (10000, 67)
Processing chunk 25: (10000, 67)
Processing chunk 26: (10000, 67)
Processing chunk 27: (10000, 67)
Processing chunk 28: (10000, 67)
Processing chunk 29: (10000, 67)
Processing chunk 30: (10000, 67)
Processing chunk 31

In [11]:
import pandas as pd

file_path = "/home/user/Downloads/bayut_uae_2026_10_15.json"
chunk_size = 10_000

issue_examples = {}

for chunk_no, chunk in enumerate(
    pd.read_json(file_path, lines=True, chunksize=chunk_size),
    start=1
):
    print(f"Processing chunk {chunk_no}")

    for col in chunk.columns:

        # Keep only the first issue per field
        if col in issue_examples:
            continue

        string_mask = chunk[col].map(lambda x: isinstance(x, str))
        values = chunk.loc[string_mask, col].astype("string")

        if values.empty:
            continue

        checks = [
            ("Leading whitespace", values.str.match(r"^\s+")),
            ("Trailing whitespace", values.str.match(r"\s+$")),
            (
                "Multiple consecutive spaces",
                values.str.contains(r" {2,}", regex=True, na=False)
            ),
            (
                "Tabs or newlines",
                values.str.contains(r"[\t\r\n]", regex=True, na=False)
            )
        ]

        for issue_type, mask in checks:
            if mask.any():
                idx = values.index[mask][0]

                record_no = (
                    (chunk_no - 1) * chunk_size
                    + chunk.index.get_loc(idx) + 1
                )

                issue_examples[col] = {
                    "record_no": record_no,
                    "url": chunk.loc[idx, "url"]
                          if "url" in chunk.columns else None,
                    "issue": issue_type
                }
                break

print("\n========== WHITESPACE QA REPORT ==========")

if issue_examples:
    report = pd.DataFrame([
        {"field": field, **details}
        for field, details in issue_examples.items()
    ])

    print(report.to_string(index=False))
    print(f"\nTotal fields with whitespace issues: {len(report)}")
else:
    print("No whitespace issues found.")

Processing chunk 1
Processing chunk 2
Processing chunk 3
Processing chunk 4
Processing chunk 5
Processing chunk 6
Processing chunk 7
Processing chunk 8
Processing chunk 9
Processing chunk 10
Processing chunk 11
Processing chunk 12
Processing chunk 13
Processing chunk 14
Processing chunk 15
Processing chunk 16
Processing chunk 17
Processing chunk 18
Processing chunk 19
Processing chunk 20
Processing chunk 21
Processing chunk 22
Processing chunk 23
Processing chunk 24
Processing chunk 25
Processing chunk 26
Processing chunk 27
Processing chunk 28
Processing chunk 29
Processing chunk 30
Processing chunk 31
Processing chunk 32
Processing chunk 33
Processing chunk 34
Processing chunk 35
Processing chunk 36
Processing chunk 37
Processing chunk 38
Processing chunk 39
Processing chunk 40
Processing chunk 41
Processing chunk 42
Processing chunk 43
Processing chunk 44
Processing chunk 45
Processing chunk 46
Processing chunk 47
Processing chunk 48
Processing chunk 49
Processing chunk 50
Processin

In [2]:
import pandas as pd
import json

file_path = "/home/user/Downloads/bayut_uae_2026_10_15.json"
chunk_size = 10_000

for chunk_no, chunk in enumerate(
    pd.read_json(file_path, lines=True, chunksize=chunk_size),
    start=1
):
    print(f"Chunk {chunk_no}: {chunk.shape}")

    # Convert list/dict columns only when needed for QA
    for col in chunk.columns:
        if chunk[col].map(lambda x: isinstance(x, (list, dict))).any():
            chunk[col] = chunk[col].map(
                lambda x: json.dumps(x, sort_keys=True, ensure_ascii=False)
                if isinstance(x, (list, dict))
                else x
            )

    # Perform your QA checks on this chunk here

Chunk 1: (10000, 67)
Chunk 2: (10000, 67)
Chunk 3: (10000, 67)
Chunk 4: (10000, 67)
Chunk 5: (10000, 67)
Chunk 6: (10000, 67)
Chunk 7: (10000, 67)
Chunk 8: (10000, 67)
Chunk 9: (10000, 67)
Chunk 10: (10000, 67)
Chunk 11: (10000, 67)
Chunk 12: (10000, 67)
Chunk 13: (10000, 67)
Chunk 14: (10000, 67)
Chunk 15: (10000, 67)
Chunk 16: (10000, 67)
Chunk 17: (10000, 67)
Chunk 18: (10000, 67)
Chunk 19: (10000, 67)
Chunk 20: (10000, 67)
Chunk 21: (10000, 67)
Chunk 22: (10000, 67)
Chunk 23: (10000, 67)
Chunk 24: (10000, 67)
Chunk 25: (10000, 67)
Chunk 26: (10000, 67)
Chunk 27: (10000, 67)
Chunk 28: (10000, 67)
Chunk 29: (10000, 67)
Chunk 30: (10000, 67)
Chunk 31: (10000, 67)
Chunk 32: (10000, 67)
Chunk 33: (10000, 67)
Chunk 34: (10000, 67)
Chunk 35: (10000, 67)
Chunk 36: (10000, 67)
Chunk 37: (10000, 67)
Chunk 38: (10000, 67)
Chunk 39: (10000, 67)
Chunk 40: (10000, 67)
Chunk 41: (10000, 67)
Chunk 42: (10000, 67)
Chunk 43: (10000, 67)
Chunk 44: (10000, 67)
Chunk 45: (10000, 67)
Chunk 46: (10000, 6

In [3]:
import pandas as pd
import json
import os
import re
import random
from datetime import datetime

# ============================================================
# 1. CONFIGURATION
# ============================================================

file_path = "/home/user/Downloads/dubizzle_uae_2026_10_02.json"

expected_fields = """
reference_number
id
url
broker_display_name
broker
category
category_url
title
description
location
price
currency
price_per
bedrooms
bathrooms
furnished
rera_permit_number
dtcm_licence
scraped_ts
amenities
details
agent_name
number_of_photos
user_id
phone_number
date
iteration_number
rera_registration_number
ded_license_number
rera_length
latitude
longitude
listing_id
package_type
locality
object_id
completion_status
ad_type
verified
sub_category_1
sub_category_2
property_type
depth
published_at
listing_availability
brn_registration_number
orn_number
bln_number
adm_number
3d_live
3d_image
2d_image
ded_number_2
ded_number_3
ded_number_4
trucheck_on
email
trubroker
rating
no_of_reviews
property_of_the_week
quality_lister
trakheesi_url
floorplan_type_identifier
floorplans_total
floorplan_id
floorplan_raw
""".strip().splitlines()

expected_empty_fields = [
    "dtcm_licence",
    "user_id",
    "listed_by",
    "ded_license_number"
]

allowed_values = {
    "price_per": {
        "", "Bi-Yearly", "Monthly", "Quarterly",
        "Yearly", "Daily", "سنويا"
    },
    "currency": {"AED"},
    "furnished": {"", "Yes"},
    "category": {"rent", "sale"},
    "package_type": {"standard", "featured", "premium"},
    "sub_category_1": {"commercial", "residential"},
    "completion_status": {"", "Ready", "Off-plan"},
    "verified": {"", "Checked", "TruCheckTM"}
}

date_fields = [
    "scraped_ts", "date", "published_at", "last_update"
]

# Update this to the expected run ID for the specific dataset.
expected_run_id = "propertyfinder-weekly:dubizzle_uae:2026junw3:qa"

# The provided iteration_number example is 2026_06_03.
iteration_pattern = re.compile(r"^\d{4}_\d{2}_\d{2}$")

chunk_size = 10_000
random_check_count = 10

output_path = file_path.replace(".json", "_qa_report.txt")

# ============================================================
# 2. HELPERS
# ============================================================

def is_empty(value):
    if value is None:
        return True
    if isinstance(value, str):
        return value.strip() == ""
    return pd.isna(value)


def is_valid_date(value):
    if is_empty(value):
        return True  # Missing values handled separately if required.

    try:
        datetime.strptime(str(value).strip(), "%Y-%m-%d")
        return True
    except ValueError:
        return False


def add_issue(issues, row_no, field, value, message):
    issues.append({
        "row_no": row_no,
        "field": field,
        "value": str(value)[:300],
        "issue": message
    })


# ============================================================
# 3. FILE NAME AND FORMAT CHECKS
# ============================================================

issues = []
summary = []

file_name = os.path.basename(file_path)

summary.append("DUBIZZLE UAE QA REPORT")
summary.append("=" * 60)
summary.append(f"File name: {file_name}")

if not file_name.endswith(".json"):
    summary.append("FAIL: File format should be JSON.")
else:
    summary.append("PASS: File extension is JSON.")

# Accepts names such as dubizzle_uae_2026_10_02.json
file_name_pattern = r"^dubizzle_uae_\d{4}_\d{2}_\d{2}\.json$"

if re.match(file_name_pattern, file_name):
    summary.append("PASS: File name follows the accepted naming pattern.")
else:
    summary.append(
        "FAIL: File name does not follow dubizzle_uae_YYYY_MM_DD.json."
    )

if "_uae_" in file_name:
    summary.append("PASS: Country suffix '_uae_' is present.")
else:
    summary.append("FAIL: Country suffix '_uae_' is missing.")

# ============================================================
# 4. CHUNK-BASED DATA QA
# ============================================================

total_rows = 0
seen_ids = set()
random_records = []
columns_found = None
chunk_count = 0

# Reservoir sampling selects 10 random records across the whole file.
rng = random.Random(42)

try:
    reader = pd.read_json(
        file_path,
        lines=True,
        chunksize=chunk_size
    )

    for chunk_no, chunk in enumerate(reader, start=1):
        chunk_count += 1
        chunk_start = total_rows
        chunk = chunk.reset_index(drop=True)

        if columns_found is None:
            columns_found = list(chunk.columns)

        total_rows += len(chunk)

        print(f"Processing chunk {chunk_no}: {len(chunk):,} rows")

        # ----------------------------------------------------
        # A. RANDOM SAMPLE OF 10 RECORDS
        # ----------------------------------------------------

        for idx, record in chunk.iterrows():
            record_no = chunk_start + idx + 2
            item = {
                "row_no": record_no,
                "record": record.to_dict()
            }

            if len(random_records) < random_check_count:
                random_records.append(item)
            else:
                position = rng.randint(0, record_no - 2)
                if position < random_check_count:
                    random_records[position] = item

        # ----------------------------------------------------
        # B. EMPTY EXPECTED FIELDS
        # ----------------------------------------------------

        for field in expected_empty_fields:
            if field not in chunk.columns:
                continue

            mask = ~chunk[field].map(is_empty)

            for idx in chunk.index[mask]:
                add_issue(
                    issues,
                    chunk_start + idx + 2,
                    field,
                    chunk.at[idx, field],
                    "Expected empty field but value is populated"
                )

        # ----------------------------------------------------
        # C. ALLOWED VALUES
        # ----------------------------------------------------

        for field, allowed in allowed_values.items():
            if field not in chunk.columns:
                continue

            for idx, value in chunk[field].items():
                if is_empty(value):
                    normalized = ""
                else:
                    normalized = str(value).strip()

                if normalized not in allowed:
                    add_issue(
                        issues,
                        chunk_start + idx + 2,
                        field,
                        value,
                        f"Unexpected value; allowed values: {sorted(allowed)}"
                    )

        # ----------------------------------------------------
        # D. BROKER AND BROKER_DISPLAY_NAME COMPARISON
        # ----------------------------------------------------

        if {"broker", "broker_display_name"}.issubset(chunk.columns):
            for idx, row in chunk.iterrows():
                broker = row["broker"]
                display = row["broker_display_name"]

                if not is_empty(broker) and not is_empty(display):
                    if str(broker).strip() != str(display).strip():
                        add_issue(
                            issues,
                            chunk_start + idx + 2,
                            "broker",
                            broker,
                            "broker and broker_display_name do not match"
                        )

                if isinstance(broker, str) and broker != broker.upper():
                    add_issue(
                        issues,
                        chunk_start + idx + 2,
                        "broker",
                        broker,
                        "broker value is not in uppercase"
                    )

        # ----------------------------------------------------
        # E. DATE FORMATS
        # ----------------------------------------------------

        for field in date_fields:
            if field not in chunk.columns:
                continue

            for idx, value in chunk[field].items():
                if not is_valid_date(value):
                    add_issue(
                        issues,
                        chunk_start + idx + 2,
                        field,
                        value,
                        "Invalid date format; expected YYYY-MM-DD"
                    )

        # ----------------------------------------------------
        # F. ITERATION NUMBER
        # ----------------------------------------------------

        if "iteration_number" in chunk.columns:
            for idx, value in chunk["iteration_number"].items():
                if not is_empty(value) and not iteration_pattern.match(
                    str(value).strip()
                ):
                    add_issue(
                        issues,
                        chunk_start + idx + 2,
                        "iteration_number",
                        value,
                        "Invalid format; expected YYYY_MM_week, e.g. 2026_06_03"
                    )

        # ----------------------------------------------------
        # G. DUPLICATE DATASET CHECK
        # ----------------------------------------------------

        if "id" in chunk.columns:
            for idx, value in chunk["id"].items():
                if is_empty(value):
                    continue

                key = str(value).strip()

                if key in seen_ids:
                    add_issue(
                        issues,
                        chunk_start + idx + 2,
                        "id",
                        value,
                        "Duplicate ID"
                    )
                else:
                    seen_ids.add(key)

        # ----------------------------------------------------
        # H. CURRENCY CHECK: AED
        # ----------------------------------------------------

        # Already covered by allowed_values.
        # Add more requirements here if needed.

# ============================================================
# 5. FINAL VALIDATION
# ============================================================

    summary.append("")
    summary.append(f"Total records: {total_rows:,}")
    summary.append(f"Total chunks processed: {chunk_count}")

    if 170_000 <= total_rows <= 190_000:
        summary.append("PASS: Record count is within the standard range.")
    elif total_rows >= 495_900:
        summary.append("INFO: Record count is within the latest dataset scale.")
    else:
        summary.append("CHECK: Record count is outside the specified ranges.")

    # --------------------------------------------------------
    # REQUIRED FIELDS AND FIELD NAME CASE
    # --------------------------------------------------------

    if columns_found is not None:
        missing_fields = sorted(set(expected_fields) - set(columns_found))
        unexpected_fields = sorted(set(columns_found) - set(expected_fields))

        if not missing_fields:
            summary.append("PASS: All expected fields are present.")
        else:
            summary.append(f"FAIL: Missing fields: {missing_fields}")

        if unexpected_fields:
            summary.append(f"CHECK: Unexpected fields: {unexpected_fields}")

        wrong_case = [
            col for col in columns_found
            if col != col.lower()
        ]

        if wrong_case:
            summary.append(f"FAIL: Field names not lowercase: {wrong_case}")
        else:
            summary.append("PASS: All field names are lowercase.")

        duplicate_column_names = (
            pd.Index(columns_found)[pd.Index(columns_found).duplicated()]
            .tolist()
        )

        if duplicate_column_names:
            summary.append(
                f"FAIL: Duplicate column names: {duplicate_column_names}"
            )

    # --------------------------------------------------------
    # UNIQUE DATASET CHECK
    # --------------------------------------------------------

    if "id" in (columns_found or []):
        if total_rows == len(seen_ids):
            summary.append("PASS: Dataset IDs are unique.")
        else:
            summary.append(
                f"FAIL: Duplicate or missing IDs detected. "
                f"Rows: {total_rows:,}; unique IDs: {len(seen_ids):,}"
            )

    # --------------------------------------------------------
    # RANDOM RECORD OUTPUT
    # --------------------------------------------------------

    summary.append("")
    summary.append("RANDOM DATA CHECK (10 RECORDS)")
    summary.append("-" * 60)

    for sample in sorted(random_records, key=lambda x: x["row_no"]):
        summary.append(f"Row {sample['row_no']}:")
        summary.append(
            json.dumps(
                sample["record"],
                ensure_ascii=False,
                default=str
            )
        )

    # --------------------------------------------------------
    # ISSUE REPORT
    # --------------------------------------------------------

    summary.append("")
    summary.append(f"Total QA issues found: {len(issues):,}")
    summary.append("")

    if issues:
        issue_df = pd.DataFrame(issues)
        issue_df.to_csv(
            file_path.replace(".json", "_qa_issues.csv"),
            index=False
        )
        summary.append(
            "Detailed issues saved to: "
            + file_path.replace(".json", "_qa_issues.csv")
        )
    else:
        summary.append("No issues found by the configured checks.")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(summary))

    print("\n".join(summary[:15]))
    print(f"\nQA summary saved to: {output_path}")

except Exception as e:
    print(f"QA failed: {type(e).__name__}: {e}")

QA failed: FileNotFoundError: File /home/user/Downloads/dubizzle_uae_2026_10_02.json does not exist


In [4]:
#===================================================================================

In [ ]:
import pandas as pd
import json

file_path = "/home/user/Downloads/bahrainfinder_bah_2026_10_03.json"

try:
    df = pd.read_json(file_path, lines=True)
except ValueError:
    try:
        df = pd.read_json(file_path, lines=False)
    except ValueError:
        with open(file_path, "r", encoding="utf-8-sig") as f:
            data = json.load(f)

        df = pd.DataFrame(data)

print(df.shape)
display(df.head())

(157, 36)


,category,bayut_property_type_id,transaction_hash_id,date_transaction_nk,builtup_area_sqm,plot_area_sqm,transaction_amount,bayut_leaf_location_name_en,bayut_leaf_location_name_ar,transaction_category_l1_name,...,bayut_location_l6_name_ar,bayut_leaf_location_id,property_completion_status_sk,transaction_per_sqm_amount,sale_market_name,date_previous_sale_transaction_nk,balcony_area_sqm,contract_monthly_amount,occupancy_status_name,has_kpi_value
0,Commercial,6,651e5d0027837ea3,2026-10-07 00:00:00,101.00,,5305750.00,Breez by Danube,بريز من دانوب,Sales,...,,22488,off-plan,52532.18,Primary First Sale,1970-01-01,32.00,,Unknown,False
1,Commercial,5,a72c6b724c15481,2026-10-09 00:00:00,65.91,,1200000.00,The Citadel,ذا سيتاديل,Sales,...,,5134,ready,18206.65,Secondary Resale,2025-01-14,0.00,,Vacant,True
2,Commercial,6,42f29b4658e83e0c,2026-10-06 00:00:00,162.12,,2294734.00,One Beverly,وان بيفرلي,Sales,...,,20465,off-plan,14154.54,Primary First Sale,1970-01-01,52.63,,Unknown,False
3,Commercial,5,56b865ef2252022d,2026-10-07 00:00:00,494.62,,36392940.00,Lumena by Omniyat,لومينا من أمنيات,Sales,...,,22046,off-plan,73577.57,Primary First Sale,1970-01-01,58.89,,Unknown,False
4,Commercial,5,5789d9629d608568,2026-10-12 00:00:00,101.52,,1049427.08,Jumeirah Bay X2,خور الجميرا X2,Sales,...,,5179,ready,10337.15,Secondary Resale,2026-02-24,0.00,,Vacant,True


In [29]:
df.shape

(157, 36)

In [30]:
df.duplicated().sum()

np.int64(0)

In [31]:
df.columns

Index(['category', 'bayut_property_type_id', 'transaction_hash_id',
       'date_transaction_nk', 'builtup_area_sqm', 'plot_area_sqm',
       'transaction_amount', 'bayut_leaf_location_name_en',
       'bayut_leaf_location_name_ar', 'transaction_category_l1_name', 'beds',
       'latitude', 'longitude', 'bayut_latitude', 'bayut_longitude', 'floor',
       'rent_contract_duration_months', 'rent_contract_renewal_status_name',
       'date_start_nk', 'bayut_location_l3_name_en',
       'bayut_location_l3_name_ar', 'bayut_location_l4_name_en',
       'bayut_location_l4_name_ar', 'bayut_location_l5_name_en',
       'bayut_location_l5_name_ar', 'bayut_location_l6_name_en',
       'bayut_location_l6_name_ar', 'bayut_leaf_location_id',
       'property_completion_status_sk', 'transaction_per_sqm_amount',
       'sale_market_name', 'date_previous_sale_transaction_nk',
       'balcony_area_sqm', 'contract_monthly_amount', 'occupancy_status_name',
       'has_kpi_value'],
      dtype='object')

In [32]:
df.category.value_counts()

category
Commercial    157
Name: count, dtype: int64

In [33]:
df.date_transaction_nk.duplicated().sum()

np.int64(145)

In [34]:
import pandas as pd

df["date_transaction_nk"] = pd.to_datetime(
    df["date_transaction_nk"], errors="coerce"
)

start_date = "2026-10-07"
end_date = "2026-10-14"

issues = df[
    ~df["date_transaction_nk"].between(start_date, end_date)
    & df["date_transaction_nk"].notna()
]

print(
    issues[["date_transaction_nk", "transaction_hash_id"]]
    .sort_values("date_transaction_nk", ascending=False)
    .to_string(index=False)
)

date_transaction_nk transaction_hash_id
         2026-10-06    42f29b4658e83e0c
         2026-10-06    5884435733de9ef0
         2026-10-06    4ca48290f8b3f12e
         2026-10-06    4478d3080bceee99
         2026-10-06    1076228fe2580891
         2026-10-06    4df6b700d9597b52
         2026-10-06    6d5de1092c126a8d
         2026-10-06    6ab781beee77f247
         2026-10-06    59762f6d38745b03
         2026-10-06    731827561918ce20
         2026-10-06    68810f538802cdd5
         2026-10-06    10923cd4daae3eaf
         2026-10-06    70284b7bc327c5f1
         2026-10-06    6b502468fb5a47a2
         2026-10-06    64164d94ca26387f
         2026-10-06    4874c1832a911696
         2026-10-06     ff9254cec2a3c94
         2026-10-06    6ba038d1e6846d95
         2026-10-06    7cbec3be03b1d5fc
         2026-10-06    572840e59bc354c3
         2026-10-06    555e9216d154e8a2
         2026-10-06    5cea74c58c6e3cab
         2026-10-06    4e91ad8a66fd0c4b
         2026-10-06     ec36721525a64fe


In [36]:
import pandas as pd

# Count actual null/NaN values
null_count = df.isna().sum()

# Count empty strings and strings containing only whitespace
empty_count = df.apply(
    lambda col: col.map(
        lambda x: isinstance(x, str) and x.strip() == ""
    ).sum()
)

# Combine counts
result = pd.DataFrame({
    "null_count": null_count,
    "empty_string_count": empty_count,
    "total_empty_or_null": null_count + empty_count
})


# Show fields containing empty or null values
result = result[result["total_empty_or_null"] > 0]
result = result.sort_values("total_empty_or_null", ascending=False)

print(result.to_string())

                               null_count  empty_string_count  total_empty_or_null
beds                                    0                 157                  157
rent_contract_duration_months           0                 157                  157
contract_monthly_amount                 0                 157                  157
bayut_location_l6_name_en               0                 152                  152
bayut_location_l6_name_ar               0                 152                  152
plot_area_sqm                           0                 130                  130
bayut_location_l5_name_en               0                  78                   78
bayut_location_l5_name_ar               0                  78                   78
floor                                   0                  27                   27
builtup_area_sqm                        0                  27                   27
bayut_location_l4_name_ar               0                   8                    8
bayu

In [26]:
# Find duplicate transaction_hash_id values
duplicates = df[df.duplicated(subset=["transaction_hash_id"], keep=False)]

print("Total duplicate records:", len(duplicates))
print("Duplicate transaction_hash_id count:",
      duplicates["transaction_hash_id"].nunique())

display(
    duplicates[["transaction_hash_id", "date_transaction_nk"]]
    .sort_values("transaction_hash_id")
)

Total duplicate records: 0
Duplicate transaction_hash_id count: 0


,transaction_hash_id,date_transaction_nk


In [5]:
#=========================================================================================================

In [1]:
import pandas as pd
import json

df = pd.read_json(
    r"/home/user/Downloads/bayut_transactions_sold_residential_2026_10_14.json",
    lines=True
)

# Convert list/dict columns into strings
df = df.apply(
    lambda col: col.map(
        lambda x: json.dumps(x, sort_keys=True)
        if isinstance(x, (list, dict))
        else x
    )
)

ValueError: Expected object or value

In [8]:
import pandas as pd

# Count null + empty string values in each column
empty_counts = {}

for col in df.columns:
    # Null values
    null_count = df[col].isna().sum()

    # Empty strings / whitespace-only strings
    empty_string_count = (
        df[col]
        .fillna('')
        .astype(str)
        .str.strip()
        .eq('')
        .sum()
    )

    # Total missing/empty
    total = null_count + empty_string_count

    empty_counts[col] = {
        'null_count': null_count,
        'empty_string_count': empty_string_count,
        'total_null_or_empty': total
    }

result = pd.DataFrame(empty_counts).T

print(result.to_string())

                      null_count  empty_string_count  total_null_or_empty
reference_number               0                3053                 3053
id                             0                   0                    0
url                            0                   0                    0
broker_display_name            0                   3                    3
broker                         0                   3                    3
category                       0                   0                    0
category_url                   0                   0                    0
title                          0                   0                    0
description                    0                   0                    0
location                       0                   0                    0
price                          0                   6                    6
currency                       0                   6                    6
price_per                      0      

In [9]:
df.furnished.value_counts()

furnished
Yes    2338
No      715
Name: count, dtype: int64

In [10]:
df.shape

(3053, 33)

In [11]:
import json

duplicate_count = (
    df.applymap(
        lambda x: json.dumps(x, sort_keys=True)
        if isinstance(x, (list, dict))
        else x
    )
    .duplicated()
    .sum()
)

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


/tmp/ipykernel_48619/600954025.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df.applymap(


In [12]:
df.duplicated().sum()

np.int64(0)

In [13]:
empty_cols = df.columns[df.isna().all()].tolist()
empty_cols

[]

In [14]:
df.isnull().sum()


reference_number        0
id                      0
url                     0
broker_display_name     0
broker                  0
category                0
category_url            0
title                   0
description             0
location                0
price                   0
currency                0
price_per               0
bedrooms                0
bathrooms               0
furnished               0
rera_permit_number      0
dtcm_licence            0
scraped_ts              0
amenities               0
details                 0
agent_name              0
number_of_photos        0
user_id                 0
phone_number            0
date                    0
iteration_number        0
depth                   0
sub_category_1          0
property_type           0
sub_category_2          0
published_at            0
listing_availability    0
dtype: int64

In [15]:
df.columns


Index(['reference_number', 'id', 'url', 'broker_display_name', 'broker',
       'category', 'category_url', 'title', 'description', 'location', 'price',
       'currency', 'price_per', 'bedrooms', 'bathrooms', 'furnished',
       'rera_permit_number', 'dtcm_licence', 'scraped_ts', 'amenities',
       'details', 'agent_name', 'number_of_photos', 'user_id', 'phone_number',
       'date', 'iteration_number', 'depth', 'sub_category_1', 'property_type',
       'sub_category_2', 'published_at', 'listing_availability'],
      dtype='object')

In [16]:
req_cols = """reference_number,
id,
url,
broker_display_name,
broker,
category,
category_url,
title,
property_type,
sub_category_1,
sub_category_2,
depth,
description,
location,
price,
currency,
price_per,
bedrooms,
bathrooms,
furnished,
rera_permit_number,
dtcm_licence,
scraped_ts,
amenities,
details,
agent_name,
number_of_photos,
user_id,
phone_number,
date,
iteration_number,
published_at,
listing_availability"""
req_cols = [c.strip() for c in req_cols.split(",") if c.strip()]

missing = [c for c in req_cols if c not in df.columns]
extra = [c for c in df.columns if c not in req_cols]

print("Requirement columns:", len(req_cols))
print("File columns:", len(df.columns))
print("Missing:", missing)
print("Extra:", extra)
print("Order matches:", req_cols == list(df.columns))

Requirement columns: 33
File columns: 33
Missing: []
Extra: []
Order matches: False


In [17]:
print(df.columns.tolist())

['reference_number', 'id', 'url', 'broker_display_name', 'broker', 'category', 'category_url', 'title', 'description', 'location', 'price', 'currency', 'price_per', 'bedrooms', 'bathrooms', 'furnished', 'rera_permit_number', 'dtcm_licence', 'scraped_ts', 'amenities', 'details', 'agent_name', 'number_of_photos', 'user_id', 'phone_number', 'date', 'iteration_number', 'depth', 'sub_category_1', 'property_type', 'sub_category_2', 'published_at', 'listing_availability']


In [18]:
broker = df["broker"].fillna("").astype(str).str.strip()
display_name = df["broker_display_name"].fillna("").astype(str).str.strip()

# Find invalid rows
invalid = df[
    broker != display_name.str.upper()
][["id", "url", "broker", "broker_display_name"]]

print(f"Invalid rows: {len(invalid)}")
display(invalid)

Invalid rows: 0


,id,url,broker,broker_display_name


In [19]:
df.published_at.value_counts()

published_at
2026-09-28    738
2026-09-21    658
2026-09-14    631
2026-10-05    409
2026-10-08    141
2026-10-11    126
2026-10-07    113
2026-10-10    110
2026-10-06     89
2026-10-09     24
2025-10-12      6
2026-09-12      4
2026-10-12      2
2026-08-12      1
2025-12-12      1
Name: count, dtype: int64

In [20]:
duplicates = df[df.duplicated(subset="url", keep=False)]

print(f"Duplicate URLs: {len(duplicates)}")
display(duplicates.sort_values("url"))

Duplicate URLs: 0


,reference_number,id,url,broker_display_name,broker,category,category_url,title,description,location,price,currency,price_per,bedrooms,bathrooms,furnished,rera_permit_number,dtcm_licence,scraped_ts,amenities,details,agent_name,number_of_photos,user_id,phone_number,date,iteration_number,depth,sub_category_1,property_type,sub_category_2,published_at,listing_availability


In [21]:
df1 = df.copy()

for col in df1.columns:
    if df1[col].apply(lambda x: isinstance(x, list)).any():
        df1[col] = df1[col].astype(str)

duplicates = df1[df1.duplicated(keep=False)]

print(f"Duplicate rows: {len(duplicates)}")
display(duplicates)

Duplicate rows: 0


,reference_number,id,url,broker_display_name,broker,category,category_url,title,description,location,price,currency,price_per,bedrooms,bathrooms,furnished,rera_permit_number,dtcm_licence,scraped_ts,amenities,details,agent_name,number_of_photos,user_id,phone_number,date,iteration_number,depth,sub_category_1,property_type,sub_category_2,published_at,listing_availability


In [22]:
import pandas as pd

df1 = df.replace(r'^\s*$', pd.NA, regex=True)

empty_columns = df1.columns[df1.isna().all()]

print("Completely empty columns:")
print(list(empty_columns))

Completely empty columns:
['reference_number', 'price_per', 'rera_permit_number', 'dtcm_licence', 'agent_name', 'user_id', 'sub_category_2']


In [23]:
import re

url_pattern = re.compile(
    r'^(https?://)'
    
    r'([a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}'
    r'(/[^\s]*)?$'
)

invalid_url = df[
    ~df["url"].fillna("").astype(str).str.match(url_pattern)
]

print(f"Invalid URLs: {len(invalid_url)}")
display(invalid_url[[ "url"]])

Invalid URLs: 0


,url


In [24]:
df.listing_availability.value_counts()

listing_availability
True     3051
False       2
Name: count, dtype: int64

In [25]:
df.dtypes

reference_number                object
id                               int64
url                             object
broker_display_name             object
broker                          object
category                        object
category_url                    object
title                           object
description                     object
location                        object
price                           object
currency                        object
price_per                       object
bedrooms                        object
bathrooms                       object
furnished                       object
rera_permit_number              object
dtcm_licence                    object
scraped_ts                      object
amenities                       object
details                         object
agent_name                      object
number_of_photos                object
user_id                         object
phone_number                    object
date                    d

In [26]:
phone_pattern = re.compile(r'^\+?\d+$')

invalid_phone = df[
    ~df["phone_number"].fillna("").astype(str).str.match(phone_pattern)
]

display(invalid_phone[["id", "phone_number"]])

,id,phone_number
45,105731646,
60,105757668,
61,105756071,
70,105767298,
80,105766749,
81,105766738,
98,105711788,
100,105635856,
129,105756170,
130,500134673,


In [27]:
df.sub_category_1.value_counts()

sub_category_1
Commercial     1866
Residential    1187
Name: count, dtype: int64

In [28]:
import pandas as pd

details = df["details"].fillna("").astype(str).str.strip()

# Must end with: number + space + M²
mask = details.ne("") & ~details.str.fullmatch(r"\d+(?:\.\d+)? M²", na=False)

result = df.loc[mask, ["url", "details"]].copy()
result["record_no"] = result.index + 2

print("Invalid details:", len(result))
print(result[["record_no", "url", "details"]].to_string(index=False))

Invalid details: 3053
 record_no                                                                                                                                                                                                                                                                                                                                                                                                                             url      details
         2                                                                                                                                                                                           https://www.dubizzle.com.bh/en/ad/%D9%84%D9%84%D8%A7%D9%8A%D8%AC%D8%A7%D8%B1-%D8%B3%D8%AA%D8%AF%D9%8A%D9%88-%D9%85%D9%81%D8%B1%D9%88%D8%B4-%D9%81%D9%8A-%D8%AC%D8%AF%D8%B9%D9%84%D9%8A-for-rent-f-f-studio-in-jidali-ID105767254.html        20 m²
         3                                                                                        

In [29]:
import re

email_pattern = re.compile(
    r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'
)

invalid_emails = df[
    df["email"].notna() &
    (df["email"].astype(str).str.strip() != "") &
    ~df["email"].astype(str).str.strip().str.match(email_pattern)
]

print(f"Invalid emails: {len(invalid_emails)}")
display(invalid_emails[["email"]])

KeyError: 'email'

In [30]:
duplicates = df[df.duplicated("id", keep=False)]

print(f"Duplicate IDs: {duplicates.shape[0]}")
display(duplicates[["id"]])

Duplicate IDs: 0


,id


In [31]:
duplicates = (
    df[df["id"].duplicated(keep=False)]
    .sort_values("id")
)

print(f"Duplicate rows: {len(duplicates)}")
display(duplicates)

Duplicate rows: 0


,reference_number,id,url,broker_display_name,broker,category,category_url,title,description,location,price,currency,price_per,bedrooms,bathrooms,furnished,rera_permit_number,dtcm_licence,scraped_ts,amenities,details,agent_name,number_of_photos,user_id,phone_number,date,iteration_number,depth,sub_category_1,property_type,sub_category_2,published_at,listing_availability


In [32]:
allowed = [
    "Apartments",
    "Rooms",
    "Villas",
    "Commercial",
    "Multiple-units",
    "Buildings",
    "Garage",
    "Chalet",
    "land",
    "Rentals-wanted"
]

invalid = df[~df["property_type"].isin(allowed)]

display(invalid[["id", "property_type"]])

,id,property_type
118,500133900,Land
119,105758935,Land
120,104795861,Land
176,500133962,Land
177,105168575,Land
178,500133899,Land
179,105462629,Land
180,105766897,Land
181,105766742,Land
182,105751021,Land


In [33]:
date_regex = r"^\d{4}-\d{2}-\d{2}$"

for col in ["scraped_ts", "date"]:
    invalid = df[
        ~df[col].fillna("").astype(str).str.match(date_regex)
    ]
    print(col, len(invalid))

scraped_ts 0
date 0


In [34]:
df.date.value_counts()

date
2026-10-12    3053
Name: count, dtype: int64

In [35]:
broker = df["broker"].fillna("").astype(str).str.strip()
display_name = df["broker_display_name"].fillna("").astype(str).str.strip()

invalid = df[
    broker != display_name.str.upper()
]

display(invalid[["id", "broker", "broker_display_name"]])

,id,broker,broker_display_name


In [36]:
df.type.value_counts()  

AttributeError: 'DataFrame' object has no attribute 'type'

In [37]:
import pandas as pd

invalid = df[
   
     df["developer_start_price"].replace(r'^\s*$', pd.NA, regex=True).isna() 
]

print(f"Rows where both prices are missing: {len(invalid)}")
display(invalid)

KeyError: 'developer_start_price'

In [38]:
df.currency.value_counts()


currency
BHD    3047
          6
Name: count, dtype: int64

In [25]:
import pandas as pd

text_cols = df.select_dtypes(include="object").columns

issues = []

for col in text_cols:
    s = df[col].fillna("").astype(str)

    mask = (
        s.str.match(r"^\s") |                  # Leading whitespace
        s.str.match(r".*\s$") |                # Trailing whitespace
        s.str.contains(r" {2,}", regex=True) | # Multiple consecutive spaces
        s.str.contains(r"[\t\r\n]", regex=True) |  # Tabs/newlines
        s.str.contains("\u00A0", regex=False)      # Non-breaking space
    )

    if mask.any():
        temp = df.loc[mask, [col]].copy()
        temp["column"] = col
        temp["row_index"] = temp.index
        issues.append(temp)

if issues:
    whitespace_issues = pd.concat(issues, ignore_index=True)
    print(f"Total issues found: {len(whitespace_issues)}")
    display(whitespace_issues[["row_index", "column"] + [c for c in whitespace_issues.columns if c not in ["row_index","column"]]])
else:
    print("✅ No unwanted whitespace found.")

Total issues found: 121423


KeyboardInterrupt: 

In [30]:
# 2. Define the columns
columns_to_analyze = [
    "category", "title", "location", "price", "currency", "price_per",
    "bedrooms", "bathrooms", "furnished", "rera_permit_number",
    "dtcm_licence", "scraped_ts", "amenities", "details", "agent_name",
    "number_of_photos", "user_id", "phone_number", "date", "iteration_number",
    "depth", "sub_category_1", "property_type", "sub_category_2",
    "published_at", "listing_availability", "ownership", "verified_business"
]

# 3. Print the number of unique entries and their breakdown
for col in columns_to_analyze:
    if col in df.columns:
        unique_count = df[col].nunique()
        print(f"==================================================")
        print(f"COLUMN: {col}")
        print(f"Total Unique Values: {unique_count}")
        print(f"==================================================")
        print(df[col].value_counts())
        print("\n")

COLUMN: category
Total Unique Values: 2
category
sale    3990
rent     627
Name: count, dtype: int64


COLUMN: title
Total Unique Values: 28
title
أرض سكنية للبيع                          2543
فيلا سكنية للبيع                          727
شقة سكنية للإيجار                         447
شقة سكنية للبيع                           288
أرض تجارية للبيع                          236
مبنى تجاري للبيع                           77
فيلا سكنية للإيجار                         59
شقة استوديو سكنية للإيجار                  55
شقة استوديو سكنية للبيع                    20
Residential Villa For Rent                 16
Residential Land For Sale                  16
Residential Villa For Sale                 16
Commercial Office For Rent                 16
Commercial Land For Sale                   16
Commercial Building For Sale               16
Residential Apartment For Sale             14
Residential Apartment For Rent             14
مكتب تجاري للإيجار                          8
Residential Penthouse For

In [40]:
df.bathrooms.value_counts()

bathrooms
       2980
2       465
5       278
4       216
3       208
6       192
1       132
7        85
8        32
9        10
10        9
12        5
11        2
375       1
17        1
14        1
Name: count, dtype: int64

In [36]:
df.nunique()

reference_number       3857
url                    4617
broker_display_name      29
broker                   29
category                  2
category_url              2
title                    28
description            4107
location                179
price                  2221
currency                  1
price_per                 1
bedrooms                  8
bathrooms                16
furnished                 7
rera_permit_number        1
dtcm_licence              1
scraped_ts                1
amenities              1464
details                1915
agent_name               95
number_of_photos          3
phone_number             53
user_id                   1
date                      1
iteration_number          1
sub_category_1            4
sub_category_2            1
property_type            13
depth                     3
last_update             124
published_at            651
id                     4617
dtype: int64

In [59]:
import pandas as pd

date_cols = [
    "scraped_ts",
    "license_start_date",
    "license_end_date",
    "date"
]

for col in date_cols:
    # Convert to datetime using the expected format
    parsed = pd.to_datetime(df[col], format="%Y-%m-%d", errors="coerce")

    invalid = df[parsed.isna() & df[col].notna() & (df[col].astype(str).str.strip() != "")]

    print(f"\n{col}")
    print(f"Invalid date format count: {len(invalid)}")

    if len(invalid) > 0:
        display(invalid[[col]])


scraped_ts
Invalid date format count: 0


KeyError: 'license_start_date'

In [60]:
df.isnull().sum()

reference_number       0
id                     0
url                    0
broker_display_name    0
broker                 0
category               0
category_url           0
title                  0
description            0
location               0
price                  0
currency               0
price_per              0
bedrooms               0
bathrooms              0
furnished              0
rera_permit_number     0
dtcm_licence           0
scraped_ts             0
amenities              0
details                0
agent_name             0
number_of_photos       0
user_id                0
phone_number           0
date                   0
iteration_number       0
depth                  0
sub_category_1         0
property_type          0
sub_category_2         0
published_at           0
dtype: int64

In [61]:
null_counts = df.isna().sum()

print(null_counts)

reference_number       0
id                     0
url                    0
broker_display_name    0
broker                 0
category               0
category_url           0
title                  0
description            0
location               0
price                  0
currency               0
price_per              0
bedrooms               0
bathrooms              0
furnished              0
rera_permit_number     0
dtcm_licence           0
scraped_ts             0
amenities              0
details                0
agent_name             0
number_of_photos       0
user_id                0
phone_number           0
date                   0
iteration_number       0
depth                  0
sub_category_1         0
property_type          0
sub_category_2         0
published_at           0
dtype: int64


In [62]:
empty_counts = (
    df.astype(str)
      .apply(lambda col: (col.str.strip() == "").sum())
)

print(empty_counts)

reference_number       2974
id                        0
url                       0
broker_display_name       0
broker                    0
category                  0
category_url              0
title                     0
description               0
location                  0
price                     5
currency                  5
price_per              2974
bedrooms                728
bathrooms               728
furnished                 0
rera_permit_number     2974
dtcm_licence           2974
scraped_ts                0
amenities               808
details                   0
agent_name             2974
number_of_photos          5
user_id                2974
phone_number            260
date                      0
iteration_number          0
depth                  2961
sub_category_1            0
property_type             0
sub_category_2         2974
published_at              0
dtype: int64


In [63]:
df[df["phone_number_1"].astype(str).str.strip() == ""]

KeyError: 'phone_number_1'

In [64]:
import json

file_path = r"/home/user/Downloads/dari_ae_2025_07_04.json"

with open(file_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        try:
            json.loads(line)
        except Exception as e:
            print(f"Error at line {i}")
            print(e)
            print(line[:500])   # show the beginning of the problematic line
            break
    else:
        print("✅ No JSON syntax errors found.")

✅ No JSON syntax errors found.


In [65]:
import json

file_path = r"/home/user/Downloads/dari_ae_2025_07_04.json"

with open(file_path, "r", encoding="utf-8") as f:
    try:
        data = json.load(f)
        print("✅ Valid JSON")
    except Exception as e:
        print(e)

Extra data: line 2 column 1 (char 498)


In [39]:
cols = ["phone_number_1", "phone_number_2"]

df[cols] = df[cols].fillna("").astype(str).apply(lambda x: x.str.strip())

same_phone = df[
    (df["phone_number_1"] != "") &
    (df["phone_number_1"] == df["phone_number_2"])
]

print("Records with same phone_number_1 and phone_number_2:", len(same_phone))

display(same_phone[["url", "name", "phone_number_1", "phone_number_2"]])

KeyError: "None of [Index(['phone_number_1', 'phone_number_2'], dtype='object')] are in the [columns]"

In [67]:
df.scraped_ts.value_counts()

scraped_ts
2026-07-20    2607
2026-07-27     367
Name: count, dtype: int64

In [68]:
df.date.value_counts()

date
2026-07-20    2607
2026-07-27     367
Name: count, dtype: int64

In [69]:
df.iteration_number.value_counts()

iteration_number
20260703    2607
20260704     367
Name: count, dtype: int64

In [72]:
df.category.value_counts()

category
rent    2935
sale      39
Name: count, dtype: int64

In [30]:
df[df.currency == "AED"]

,reference_number,url,broker_display_name,broker,category,category_url,title,description,location,price,currency,price_per,bedrooms,bathrooms,furnished,rera_permit_number,dtcm_licence,scraped_ts,amenities,details,agent_name,number_of_photos,phone_number,user_id,date,iteration_number,sub_category_1,sub_category_2,property_type,depth,last_update,published_at,id
4123,,https://bahrainfinder.bh/ar/property/أرض-للبيع-في-كرانة-3519.html,فجر الخليج العقارية,فجر الخليج العقارية,sale,https://bahrainfinder.bh/en/for-sale/,أرض سكنية للبيع,"أرض سكنية للبيع في كرانهبتصنيف RB (مناطق السكن الخاص (ب))تقع على شارع واحدمساحة أرض 312.9 متر مربعبسعر 31.77د.ب لكل قدم مربعالسعر الإجمالي 107,003.266د.ب قابل للتفاوض",كرانة,"107,003",AED,,,,,,,2026-07-27,,312.9 م²,حسين الخياط,1,+97338100114,,2026-07-27,20260704,سكنية,,أرض,,2026-06-14,2025-02-02,18d0227a284a4dd1f27715facce65b1f


In [4]:
df.furnished.value_counts()

furnished
No     191168
Yes     43079
Name: count, dtype: int64

In [ ]:
import pandas as pd

for col in df.columns:
    print("\n" + "="*80)
    print(f"Column: {col}")
    print("="*80)

    value_counts = (
        df[col]
        .fillna("<NULL>")        # Show nulls explicitly
        .astype(str)
        .value_counts(dropna=False)
    )

    print(value_counts)

In [ ]:
df.scraped_ts.value_counts()

scraped_ts
2026-07-27    109714
2026-07-26     91308
2026-07-25     33225
Name: count, dtype: int64

: 

In [3]:
df.date.value_counts()

date
2026-07-27    109714
2026-07-26     91308
2026-07-25     33225
Name: count, dtype: int64

In [ ]:
df.published_at.value_counts()

published_at
2026-07-13    26445
2026-07-06    26319
2026-06-29    23844
2026-07-12    22644
2026-07-05    15626
2026-07-20    15087
2026-07-19    14886
2026-06-28    13025
2026-07-22    10499
2026-07-21     9790
2026-07-23     7126
2026-07-11     6593
2026-07-25     5823
2026-06-27     5436
2026-07-18     4937
2026-07-04     3813
2026-07-24     3797
2026-07-27     3394
2026-07-26     3246
2026-06-26     2961
2026-04-27     2030
2026-05-27     1787
2026-04-26     1250
2026-05-26      985
2026-03-27      663
2026-06-25      454
2026-03-26      349
2026-04-25      329
2026-05-25      327
2024-07-27      113
2025-12-27       74
2024-07-26       66
2025-07-27       60
2025-07-26       55
2026-02-27       50
2025-08-26       44
2026-03-25       42
2026-02-26       33
2025-12-26       32
2025-10-27       30
2025-08-27       29
2025-09-26       25
2025-07-25       21
2026-01-26       16
2025-11-26       15
2025-09-27       12
2025-10-26       10
2026-01-27        9
2025-12-25        9
2024-07

: 

In [37]:
df.depth.value_counts()

depth
            4576
Featured      40
مميّز          1
Name: count, dtype: int64

In [38]:
df.dtypes

reference_number               object
url                            object
broker_display_name            object
broker                         object
category                       object
category_url                   object
title                          object
description                    object
location                       object
price                          object
currency                       object
price_per                      object
bedrooms                       object
bathrooms                      object
furnished                      object
rera_permit_number             object
dtcm_licence                   object
scraped_ts                     object
amenities                      object
details                        object
agent_name                     object
number_of_photos                int64
phone_number                   object
user_id                        object
date                   datetime64[ns]
iteration_number                int64
sub_category

In [21]:
import re
import pandas as pd

issues = []

for col in df.columns:

    s = df[col].fillna("").astype(str)

    # Check leading, trailing, multiple spaces, tab, newline
    mask = (
        s.str.contains(r"^\s", regex=True) |
        s.str.contains(r"\s$", regex=True) |
        s.str.contains(r" {2,}", regex=True) |
        s.str.contains(r"\t", regex=True) |
        s.str.contains(r"\n", regex=True) |
        s.str.contains(r"\r", regex=True)
    )

    # Only the first issue from each field
    if mask.any():
        idx = df.index[mask][0]

        issues.append({
            "record_no": idx + 1,
            "url": df.loc[idx, "url"],
            "issue_field": col
        })

result = pd.DataFrame(issues)

print("Fields with whitespace issues:", len(result))
print(result.to_string(index=False))

Fields with whitespace issues: 4
 record_no                                                                                                                                                                                                                                                                                url         issue_field
       425                                                                                                                              https://www.dubizzle.com.bh/en/ad/%D8%B4%D9%82%D9%87-%D9%85%D9%81%D8%B1%D9%88%D8%B4%D8%A9-%D9%84%D9%84%D8%A7%D9%8A%D8%AC%D8%A7%D8%B1-ID105759706.html broker_display_name
       425                                                                                                                              https://www.dubizzle.com.bh/en/ad/%D8%B4%D9%82%D9%87-%D9%85%D9%81%D8%B1%D9%88%D8%B4%D8%A9-%D9%84%D9%84%D8%A7%D9%8A%D8%AC%D8%A7%D8%B1-ID105759706.html              broker
        11 https://www.dubizzle.com.bh/en/ad/%D8%

In [30]:
df.number_of_photos.value_counts()

number_of_photos
3      55096
1      34356
4      31487
5      26858
6      22493
2      19325
7      16996
8      13950
9      12050
10     11748
11      7083
12      6947
13      3203
14      2881
15      2180
16      1676
        1278
18      1235
17      1094
20       991
19       622
24       569
21       480
22       364
23       326
25       284
36       281
30       265
28       222
27       203
26       201
40       155
32       131
29       117
33        99
35        97
37        85
31        82
38        62
34        60
48        50
39        45
42        45
50        37
45        35
44        30
49        27
41        25
54        21
46        19
47        14
60        14
55        14
70        13
43        13
52        12
65        12
72        12
56        11
57        10
63         9
90         9
80         8
51         8
53         8
64         8
67         7
58         6
59         5
84         5
100        5
81         5
66         5
68         4
135        4
61      

In [29]:
df[df.bathrooms== "2.5"].url

81219    https://aqarmap.com.eg/en/listing/7155474-for-sale-cairo-dokki-mydn-lmsh-almosekar-ali-ismail-st/
Name: url, dtype: object